# Interactive Tutorial for the HWO High-Resolution Imager (HRI)

[SYOTools](https://github.com/spacetelescope/hwo-tools) is a framework to enable users to create [Science Yield Optimization web tools](http://hwo.stsci.edu) for observatory design. It uses Bokeh & astropy to visualize exposure time calculators (ETCs), as well as other science cases, as a function of various observatory parameters. While SYOTools can be used independently of a particular observatory or science case, it has been primarily written to facilitate the design of [HWO](https://www.habitableworldsobservatory.org).

SYOTools is divided into two main parts, contained in the `syotools.models` and `syotools.interface` subpackages. This Jupyter Notebook is intended as a walkthrough for using `syotools.models` to perform calculations. This now can and should be modified to suit the users needs.

This notebook uses `synphot`, which require some reference files to work. Installation instructions are beyond the scope of this work but instructions and links can be found in the [synphot Installation and Setup page](https://stsynphot.readthedocs.io/en/latest/#installation-and-setup)

In [ ]:
#Import Bokeh interface tools
from ipywidgets import interact
from bokeh.io import push_notebook, show, output_notebook
from bokeh.plotting import figure
from bokeh.layouts import column
output_notebook()

#Import numpy, syotools, and astropy.units
### NOTE: You may receive some synphot warnings when executing these imports, these are expected
import numpy as np
from syotools.models import Telescope, Source #models for the observatory and instruments
from syotools.utils.jsonunit import str_jsunit #for printing JsonUnit and JsonSpectrum wrappers in a readable way
from syotools.spectra.spec_defaults import syn_spectra_library
import astropy.units as u
import synphot as syn

#make sure that synphot can find its reference files!
import os
assert os.path.exists(os.environ["PYSYN_CDBS"])

## Initialization

*"The High Resolution Imager (HRI) instrument is the primary astronomical imaging instrument for observations in the near UV through the near IR. The hri design provides a 2 x 3 arcminute field-of-view, taking full advantage of the angular resolution provided by the telescope, and consists of two channels - an ultraviolet-visible (UVIS) channel covering 200 nm - 950 nm and a near-infrared (NIR) channel covering the range 800 nm - 2200 nm. The respective focal plane detector arrays provide Nyquist sampled images at 400 nm (2.73 mas/pixel) for UVIS imaging and at 1200 nm (8.20 mas/pixel) for NIR imaging."* (adapted from LUVOIR STDT study) 

In this example, we will create a exposure time calculator for HRI, so that we can calculate the signal-to-noise ratio (SNR) for several possible template spectra. This expands on the capabilities provided by [the official HRI ETC tool](http://hwo.stsci.edu/camera_etc), adding the following options:

1. Customize the range of the primary aperture.
2. Select a band to use to normalize the spectral template.

In [ ]:
#Instatiate the observatory using syotools.models.Telescope
#This loads the default Telescope values, which are already based on the HWO design. 
#It also now loads all of the instruments (Cameras, Spectrographs, and IFSes) into a variable.
hwo = Telescope()
hwo.set_from_hwome('EAC5')

Let's print the list of available instruments.

In [ ]:
# see all of the instruments
for instrument in hwo.instruments.keys():
    print(instrument)

We will make two objects, one for each of the HRI channels.

In [ ]:
hri_uvis = hwo.instruments["HRI_S.HRI_S_UVIS_Imager"]
hri_ir = hwo.instruments["HRI_S.HRI_S_NIR_Imager"]

Now we will set up an exposure and the type of calculation we want to simulate.

In [ ]:
#Create a new photometric exposure for the camera
hri_uvis_exposure = hri_uvis.create_exposure()
hri_ir_exposure = hri_ir.create_exposure()

# set the kind of calculation you want done
hri_uvis_exposure.unknown = "snr"
hri_ir_exposure.unknown = "snr"



## Available options

We will now print out lists of available bands and spectral templates. For the channels, we print out the name and pivot wavelength.

In [ ]:
#Print the available wave bands
bandpass = hri_uvis.recover('bands')

pivotwave_uvis = hri_uvis.pivotwave.value
pivotwave_ir = hri_ir.pivotwave.value

print("Bands availabe in the UVIS channel:")
for band in hri_uvis.bands:
    print("   {:30s} | {:5.2f} ± {:5.2f}".format(band, hri_uvis.configuration["band"][band]["bandpass"].pivot(), hri_uvis.configuration["band"][band]["bandwidth"])) 
print("\nBands availabe in the IR channel:")
for band in hri_ir.bands:
    print("   {:30s} | {:5.2f} ± {:5.2f}".format(band, hri_ir.configuration["band"][band]["bandpass"].pivot(), hri_ir.configuration["band"][band]["bandwidth"])) 

available_hri_templates = syn_spectra_library.keys()
print("\nAvailalbe spectral templates:")
for spec in available_hri_templates:
    print(spec)

#Print the default template
default_hri_template = hri_uvis_exposure.source.name
print("\nCurrent SED template: {}".format(default_hri_template)) 

## Figure setup

Here we set up the widgets that we will be using to visualize the simulations. The first two code blocks set up the initial SED abd SNR figures.
The SNR figure plots the calculated values against the pivot wavelengths.

In [ ]:
#Create the Bokeh SED figure
hri_uvis_sed = hri_uvis_exposure.recover('source').sed
hri_uvis_wave = hri_uvis_sed.waveset.to_value(u.nm)
hri_uvis_flux = syn.units.convert_flux(hri_uvis_wave, hri_uvis_sed(hri_uvis_wave), u.STmag)

hri_ir_sed = hri_ir_exposure.recover('source').sed
hri_ir_wave = hri_ir_sed.waveset.to_value(u.nm)
hri_ir_flux = syn.units.convert_flux(hri_ir_wave, hri_ir_sed(hri_ir_wave), u.STmag)

hri_sed_fig = figure(height=300, width=600, title="SED", x_axis_label="Wavelength [nm]",
                 y_axis_label="ST Mag", y_range=(32, 20), x_range=(800, 24000))

hri_uvis_sed_line = hri_sed_fig.line(hri_uvis_wave, hri_uvis_flux, color='blue', line_width=3)
hri_ir_sed_line = hri_sed_fig.line(hri_ir_wave, hri_ir_flux, color='orange', line_width=3)

In [ ]:
#Create the Bokeh SNR figure
hri_uvis_snr = hri_uvis_exposure.recover("snr")
hri_ir_snr = hri_ir_exposure.recover("snr")

hri_uvis_snr = [hri_uvis_snr[x].value for x in range(len(hri_uvis_snr))]    
hri_ir_snr = [hri_ir_snr[x].value for x in range(len(hri_ir_snr))]    

hri_snr_fig = figure(height=300, width=600, title="SNR", x_axis_label="Wavelength [nm]", 
                 y_axis_label="Signal to noise ratio", y_range=(0, 20), x_range=(800, 24000))

hri_uvis_snr_scatter = hri_snr_fig.scatter(pivotwave_uvis, hri_uvis_snr, color='blue')
hri_ir_snr_scatter = hri_snr_fig.scatter(pivotwave_ir, hri_ir_snr, color='orange')

## Figure updater

The next code block updates the plots whenever the widget is updated. 

In [ ]:
#Define the update callback function for interactive inputs
def hri_update(template=default_hri_template, aperture=2.4, exptime=2.5, v_magnitude=25.):
    #find the correct template code
    sed_id = template

    #turn off calculations until everything is updated
    hri_uvis_exposure.disable()
    hri_ir_exposure.disable()
    
    #update all of the telescope & exposure parameters
    newsource = Source()
    newsource.set_sed(template, v_magnitude, 0, 0, bandpass="johnson,v")

    hri_uvis_exposure.telescope.effective_diameter = aperture * u.m
    hri_ir_exposure.telescope.effective_diameter = aperture * u.m

    exptime_uvis = [exptime] * len(hri_uvis.bandnames) 
    hri_uvis_exposure.exptime = exptime_uvis * u.hr
    hri_uvis_exposure.source = newsource

    exptime_ir = [exptime] * len(hri_ir.bandnames)
    hri_ir_exposure.exptime = exptime_ir * u.hr
    hri_ir_exposure.source = newsource
    
    #turn calculations back on and recalculate based on updated parameters
    hri_uvis_exposure.enable()
    hri_ir_exposure.enable()

    print('\nUsing the following parameters')
    print('==============================')
    print(f'Template:      {hri_uvis_exposure.source.name} normalized to V-magnitude {hri_uvis_exposure.source.magnitude:.2f}')
    print(f'Aperture:      {hri_uvis_exposure.telescope.effective_diameter:.2f}')
    print(f'exposure time: {hri_uvis_exposure.exptime[0]:.2f}')
    
    #recover the recalculated values, and make sure everything is in the right units
    hri_uvis_sed = hri_uvis_exposure.recover('source')
    hri_uvis_snr = hri_uvis_exposure.recover("snr")
    hri_uvis_snr = [hri_uvis_snr[x].value for x in range(len(hri_uvis_snr))]
    hri_uvis_sed = hri_uvis_sed.sed
    hri_uvis_wave = hri_uvis_sed.waveset.to_value(u.nm)
    hri_uvis_flux = syn.units.convert_flux(hri_uvis_wave, hri_uvis_sed(hri_uvis_wave), u.STmag)

    hri_ir_sed = hri_ir_exposure.recover('source')
    hri_ir_snr = hri_ir_exposure.recover("snr")
    hri_ir_snr = [hri_ir_snr[x].value for x in range(len(hri_ir_snr))]
    hri_ir_sed = hri_ir_sed.sed
    hri_ir_wave = hri_ir_sed.waveset.to_value(u.nm)
    hri_ir_flux = syn.units.convert_flux(hri_ir_wave, hri_ir_sed(hri_ir_wave), u.STmag)  
    
    #sanitize the sed fluxes because some of the synphot spectra don't play nice
    hri_uvis_flux[~np.isfinite(hri_uvis_flux)] = v_magnitude * u.STmag
    hri_ir_flux[~np.isfinite(hri_ir_flux)] = v_magnitude * u.STmag
    
    #update the SED figure
    ymax = max(hri_uvis_flux.max().value, hri_ir_flux.max().value)
    ymin = min(hri_uvis_flux.min().value, hri_ir_flux.min().value)
    hri_sed_fig.y_range.start = ymax + 1.5
    hri_sed_fig.y_range.end = ymin - 1.5
    hri_uvis_sed_line.data_source.data = {'x': hri_uvis_wave, 'y': hri_uvis_flux}
    hri_ir_sed_line.data_source.data = {'x': hri_ir_wave, 'y': hri_ir_flux}
    
    #update the SNR figure
    hri_snr_fig.y_range.start = 0.
    snr_max = max(np.max(hri_uvis_snr), np.max(hri_ir_snr))
    hri_snr_fig.y_range.end = max(1.05 * snr_max, 5.)
    hri_uvis_snr_scatter.data_source.data['y'] = hri_uvis_snr
    hri_ir_snr_scatter.data_source.data['y'] = hri_ir_snr
    
    #update the plots
    push_notebook()

## Show the plots

Depending on your jupyter config you may need to shift+enter the following cell after changing the sliders below to see an updated result plot

In [ ]:
hri_handle = show(column(hri_sed_fig, hri_snr_fig), notebook_handle=True)

## Create the interactive inputs

Press shift+enter here will reset to defaults.  

Warning: Some SED and mode combinations don't work because of incomplete spectra. If you encounter problems reset the plot by running the following cell and choosing a different combination.

In [ ]:
hri_inputs = interact(hri_update, template=available_hri_templates, v_magnitude=(20.0, 35.0, 0.1), aperture=(2.0, 12.0), exptime=(0.1, 10.0, 0.1))